# 3.1.4, Three identities derived from the Cartan relations (standard + tilde)

**Goal.** Syntactically close the following three identities, which
are *derivable* from the six core Cartan relations closed in 3.1.2 and
3.1.3, first in **standard** Cartan calculus, then on the **tilde**
(Koszul) side, on generic test objects:

$$
\begin{aligned}
& \mathcal{L}_U \mathcal{L}_V \mu - \mathcal{L}_V \mathcal{L}_U \mu - \mathcal{L}_{[U,V]} \mu = 0, \\
& \mathcal{L}_U\, d\,\iota_W \eta - d\,\iota_{[U,W]} \eta - d\,\iota_W\,\mathcal{L}_U \eta = 0, \\
& \mathcal{L}_W\, d\,\iota_V \omega - d\,\iota_W\, d\,\iota_V \omega = 0.
\end{aligned}
$$

All three are syntactic consequences of the six core relations:

| # | Standard proof chain | Tilde proof chain | Poisson? |
|---|---|---|---|
| 1 | rel-7 ($[\mathcal{L},\mathcal{L}] = \mathcal{L}_{[\cdot]}$) directly | rel-6 ($[\tilde{\mathcal{L}},\tilde{\mathcal{L}}] = \tilde{\mathcal{L}}_{[\cdot]_K}$) directly | tilde: **yes** |
| 2 | rel-5 ($[\mathcal{L},d]=0$) + rel-4 ($[\mathcal{L},\iota] = \iota_{[\cdot]}$) chain | the same chain on the tilde side: rel-5 ($[\tilde{\mathcal{L}},\tilde{d}] = 0$) + rel-4 ($[\tilde{\mathcal{L}},\tilde{\iota}] = \tilde{\iota}_{[\cdot]_K}$) | tilde: **yes** |
| 3 | Cartan magic + $d^2 = 0$ | tilde Cartan magic + $\tilde{d}^2 = 0$ | tilde: **yes** |


## Strategy

**Standard side.** Uses `engine_of` from 3.1.2 (the 1-form intrinsic
engine + 3 closure axioms + 3 0-form collapse rules); $\mu, \eta,
\omega$ are generic 1-forms, $Y$ a generic vector field, and the
proof goes via
`prove_intrinsic_equivalence(LHS, 0, engine=engine_of, …)`.

**Tilde side.** The 28-rule
`KoszulProblem.tilde_intrinsic_engine()` from 3.1.3 plus the
recognizer extension added in Phase 14.H:
`TildeSnJacobiResidueDefinition` now matches both polarities, the
canonical one (seen in rel-4/rel-6) and the overall-sign-flipped one
(the version with all signs reversed that the T2/T3 derived
identities leave under a `MultiEval(V, d(·), γ)` wrapper). Thanks to
this, the third and second identities close on a deg-2 V via a single
`prob.prove_tilde_cartan(LHS, 0, etas=(η, ξ))` call.

The first identity uses V as 1-VF and a single $\eta$ (the same
setup as rel-6); identities 2 and 3 close with V as 2-VF and
$(\eta, \xi)$, when V is 1-VF, $\tilde{\iota}_\gamma V$ drops to
a 0-form and the reduction `ι̃_γ(d̃ f)` is not among the engine's
natural rules.


In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401


## 1. Setup, standard-side engine + tilde-side helper

**Standard symbols:**

- $\mu, \eta, \omega$: generic 1-forms.
- $U, V, W, Y$: degree-0 vector fields (`Derivation`).

**Tilde side:** a separate `KoszulProblem` instance per identity,
all three need `assume_poisson()` ($[\pi,\pi]_{\mathrm{SN}} = 0$).


In [2]:
# --- standard side ---
from jacopy.algebra.derivation import Act, Derivation
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.intrinsic_engine import (
    intrinsic_engine_with_closure,
    prove_intrinsic_equivalence,
)
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.multi_eval import multi_eval
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.proof.expansion import (
    ExpansionEngine,
    IotaOnZeroFormDefinition,
    IotaSquaredZeroDefinition,
    LieDerivativeOnZeroFormDefinition,
)

reg = PropertyRegistry()
mu = Symbol("μ");    reg.declare(mu, Graded(degree=1))
eta = Symbol("η");   reg.declare(eta, Graded(degree=1))
omega = Symbol("ω"); reg.declare(omega, Graded(degree=1))
U = Derivation("U", 0)
V_vf = Derivation("V", 0)
W = Derivation("W", 0)
Y = Derivation("Y", 0)

defs = list(intrinsic_engine_with_closure().definitions)
defs += [
    IotaOnZeroFormDefinition(registry=reg),
    IotaSquaredZeroDefinition(),
    LieDerivativeOnZeroFormDefinition(registry=reg),
]
engine_of = ExpansionEngine(defs)


def prove_std(label, lhs, rhs):
    chain = prove_intrinsic_equivalence(lhs, rhs, engine=engine_of, registry=reg)
    print(f"{label} -> closed in {len(chain)} steps")
    return chain


# --- tilde side ---
from jacopy.calculus.tilde import (
    TildeExteriorDerivative,
    TildeInteriorProduct,
    TildeLieDerivative,
)
from jacopy.library.koszul_problem import KoszulProblem


def build_tilde(form_names, V_degree=1):
    r = PropertyRegistry()
    pi = Symbol("π")
    forms = tuple(Symbol(n) for n in form_names)
    for f in forms:
        r.declare(f, Graded(degree=1))
    Vmv = Symbol("V")
    r.declare(Vmv, Graded(degree=V_degree))
    prob = KoszulProblem(pi, forms, registry=r, multivectors=((Vmv, V_degree),))
    prob.assume_poisson()
    return (prob, Vmv) + forms


def prove_tilde(label, prob, lhs, rhs, etas):
    chain = prob.prove_tilde_cartan(lhs, rhs, etas=etas)
    print(f"{label} -> closed in {len(chain)} steps")
    return chain


print("Standard engine rule count :", len(engine_of.definitions))
_probe, *_ = build_tilde(("a", "b"))
print("Tilde engine rule count    :", len(_probe.tilde_intrinsic_engine().definitions))


Standard engine rule count : 15
Tilde engine rule count    : 28


## 2. Identity 1: $[\mathcal{L}_U, \mathcal{L}_V]\,\mu = \mathcal{L}_{[U,V]}\,\mu$

$$
\mathcal{L}_U \mathcal{L}_V \mu - \mathcal{L}_V \mathcal{L}_U \mu - \mathcal{L}_{[U,V]} \mu \;=\; 0.
$$

Directly rel-7 evaluated on $\mu$, for the 1-form $\mu$ each of
the three terms is a 1-form, evaluated against $Y$ to drop to a
scalar, and closes via the commutator + Jacobi closure axioms.


In [3]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(U), Act(lie_derivative(V_vf), mu)), Y),
    Neg(multi_eval(Act(lie_derivative(V_vf), Act(lie_derivative(U), mu)), Y)),
    Neg(multi_eval(Act(lie_derivative(lie_bracket_vf(U, V_vf)), mu), Y)),
)
chain = prove_std("(STD-1) (L_U L_V − L_V L_U − L_[U,V]) μ = 0  on μ(Y)",
                  lhs, Integer(0))
display_chain(chain)


(STD-1) (L_U L_V − L_V L_U − L_[U,V]) μ = 0  on μ(Y) -> closed in 12 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(L_U\!\left(L_V\!\left(\mu\right)\right)\right)\!\left(Y\right) \to U\!\left(\left(L_V\!\left(\mu\right)\right)\!\left(Y\right)\right) - \left(L_V\!\left(\mu\right)\right)\!\left([U,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_V\!\left(\mu\right)\right)\!\left(Y\right) \to V\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([V,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_V\!\left(\mu\right)\right)\!\left([U,Y]_{VF}\right) \to V\!\left(\mu\!\left([U,Y]_{VF}\right)\right) - \mu\!\left([V,[U,Y]_{VF]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_V\!\left(L_U\!\left(\mu\right)\right)\right)\!\left(Y\right) \to V\!\left(\left(L_U\!\left(\mu\right)\right)\!\left(Y\right)\right) - \left(L_U\!\left(\mu\right)\right)\!\left([V,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_U\!\left(\mu\right)\right)\!\left(Y\right) \to U\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([U,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_U\!\left(\mu\right)\right)\!\left([V,Y]_{VF}\right) \to U\!\left(\mu\!\left([V,Y]_{VF}\right)\right) - \mu\!\left([U,[V,Y]_{VF]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(L_{[U,V]_{VF}}\!\left(\mu\right)\right)\!\left(Y\right) \to [U,V]_{VF}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([[U,V]_{VF,Y]_{VF}}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(\left(U\!\left(V\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([V,Y]_{VF}\right)\right) - \left(V\!\left(\mu\!\left([U,Y]_{VF}\right)\right) - \mu\!\left([V,[U,Y]_{VF]_{VF}}\right)\right)\right) - \left(V\!\left(U\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([U,Y]_{VF}\right)\right) - \left(U\!\left(\mu\!\left([V,Y]_{VF}\right)\right) - \mu\!\left([U,[V,Y]_{VF]_{VF}}\right)\right)\right) - \left([U,V]_{VF}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([[U,V]_{VF,Y]_{VF}}\right)\right)\right) - 0 \to \left(\left(\left(U\!\left(V\!\left(\mu\!\left(Y\right)\right)\right) - U\!\left(\mu\!\left([V,Y]_{VF}\right)\right)\right) - \left(V\!\left(\mu\!\left([U,Y]_{VF}\right)\right) - \mu\!\left([V,[U,Y]_{VF]_{VF}}\right)\right)\right) - \left(\left(V\!\left(U\!\left(\mu\!\left(Y\right)\right)\right) - V\!\left(\mu\!\left([U,Y]_{VF}\right)\right)\right) - \left(U\!\left(\mu\!\left([V,Y]_{VF}\right)\right) - \mu\!\left([U,[V,Y]_{VF]_{VF}}\right)\right)\right) - \left([U,V]_{VF}\!\left(\mu\!\left(Y\right)\right) - \mu\!\left([[U,V]_{VF,Y]_{VF

In [4]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(L_V(μ))(Y)
 -> (U(L_V(μ)(Y)) + (-L_V(μ)([U,Y]_VF)))

[2] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_V(μ)(Y)
 -> (V(μ(Y)) + (-μ([V,Y]_VF)))

[3] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_V(μ)([U,Y]_VF)
 -> (V(μ([U,Y]_VF)) + (-μ([V,[U,Y]_VF]_VF)))

[4] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_V(L_U(μ))(Y)
 -> (V(L_U(μ)(Y)) + (-L_U(μ)([V,Y]_VF)))

[5] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(μ)(Y)
 -> (U(μ(Y)) + (-μ([U,Y]_VF)))

[6] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(μ)([V,Y]_VF)
 -> (U(μ([V,Y]_VF)) + (-μ([U,[V,Y]_VF]_VF)))

[7] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_[U,V]_VF(μ)(Y)
 -> ([U,V]_VF(μ(Y)) + (-μ([[U,V]_VF,Y]_VF)))

[8] 

### 2.T. Tilde analogue

$$
\tilde{\mathcal{L}}_\alpha \tilde{\mathcal{L}}_\beta V
   - \tilde{\mathcal{L}}_\beta \tilde{\mathcal{L}}_\alpha V
   - \tilde{\mathcal{L}}_{[\alpha,\beta]_K} V \;=\; 0.
$$

The exact same setup as rel-6 (Phase 14.G): V as 1-VF + a single
$\eta$ evaluation; under the Poisson assumption, the 14.G pipeline
already recognises this polarity.


In [5]:
prob, Vmv, alpha, beta = build_tilde(("α", "β"))
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(TildeLieDerivative(beta, prob.pi), Vmv)),
    Neg(Act(TildeLieDerivative(beta, prob.pi),
            Act(TildeLieDerivative(alpha, prob.pi), Vmv))),
    Neg(Act(TildeLieDerivative(prob.bracket(alpha, beta), prob.pi), Vmv)),
)
chain = prove_tilde(
    "(TIL-1) (L̃_α L̃_β − L̃_β L̃_α − L̃_[α,β]_K) V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t,))
display_chain(chain)


(TIL-1) (L̃_α L̃_β − L̃_β L̃_α − L̃_[α,β]_K) V = 0  [Poisson] -> closed in 120 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right) - \tilde{\mathcal{L}}_{\left[\alpha,\, \beta\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\!\left(\eta\right) \to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\!\left(\eta\right) - \left(\tilde{\mathcal{L}}_{\left[\alpha,\, \beta\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\!\left(\eta\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) \to \left(\pi\sharp\!\left(\beta\right)\right)\!\left(V\!\left(\eta\right)\right) - V\!\left(\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left[\beta,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\beta\right) - d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) \to V\!\left(L_\pi\sharp(\beta)\!\left(\eta\right)\right) - V\!\left(L_\pi\sharp(\eta)\!\left(\beta\right)\right) - V\!\left(d\!\left(\langle \pi\sharp\!\left(\beta\right),\, \eta \rangle\right)\right) \quad \text{[MultiEval arg slot linearity]\,(axiom)} \\
\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\alpha)\!\left(\eta\right) - L_\pi\sharp(\eta)\!\left(\alpha\right) - d\!\left(\langle \pi\sharp\!\left(\alpha\right),\, \eta \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\beta}\!\left(V\right)\right)\!\left(L_\pi\sharp(\alpha)\!\left(\eta\right

In [6]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_α(L̃_β(V)) + (-L̃_β(L̃_α(V))) + (-L̃_[·,·]_K[π](α, β)(V)))(η)
 -> (L̃_α(L̃_β(V))(η) + (-L̃_β(L̃_α(V))(η)) + (-L̃_[·,·]_K[π](α, β)(V)(η)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_α(L̃_β(V))(η)
 -> (π♯(α)(L̃_β(V)(η)) + (-L̃_β(V)([·,·]_K[π](α, η))))

[3] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_β(V)(η)
 -> (π♯(β)(V(η)) + (-V([·,·]_K[π](β, η))))

[4] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](β, η)
 -> (L_π♯(β)(η) + (-L_π♯(η)(β)) + (-d(⟨π♯(β), η⟩)))

[5] MultiEval arg slot linearity [axiom]
    V((L_π♯(β)(η) + (-L_π♯(η)(β)) + (-d(⟨π♯(β), η⟩))))
 -> (V(L_π♯(β)(η)) + (-V(L_π♯(η)(β))) + (-V(d(⟨π♯(β), η⟩))))

[6] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](α, η)
 -> (L_π♯(α)(η) + (-L_π♯(η)(α)) + (-d(⟨π♯(α), η⟩)))

[7] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·

## 3. Identity 2: $\mathcal{L}_U\,d\,\iota_W\,\eta = d\,\iota_{[U,W]}\,\eta + d\,\iota_W\,\mathcal{L}_U\,\eta$

$$
\mathcal{L}_U\, d\,\iota_W \eta - d\,\iota_{[U,W]} \eta - d\,\iota_W\,\mathcal{L}_U \eta \;=\; 0.
$$

Derivation: $[\mathcal{L}_U, d] = 0$ (rel-5) gives
$\mathcal{L}_U\,d = d\,\mathcal{L}_U$; then
$[\mathcal{L}_U, \iota_W] = \iota_{[U,W]}$ (rel-4) yields
$\mathcal{L}_U\,\iota_W \eta = \iota_{[U,W]} \eta + \iota_W\,\mathcal{L}_U \eta$.
Applying $d$ recovers the equality. For the 1-form $\eta$ evaluated
against $Y$, all three terms are 1-forms that drop to a scalar.


In [7]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(U), Act(default_d, Act(interior(W), eta))), Y),
    Neg(multi_eval(Act(default_d, Act(interior(lie_bracket_vf(U, W)), eta)), Y)),
    Neg(multi_eval(Act(default_d, Act(interior(W), Act(lie_derivative(U), eta))), Y)),
)
chain = prove_std(
    "(STD-2) L_U d ι_W η − d ι_[U,W] η − d ι_W L_U η = 0  on η(Y)",
    lhs, Integer(0))
display_chain(chain)


(STD-2) L_U d ι_W η − d ι_[U,W] η − d ι_W L_U η = 0  on η(Y) -> closed in 14 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(L_U\!\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) \to U\!\left(\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left(Y\right)\right) - \left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left([U,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_W\!\left(\eta\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_W\!\left(\eta\right)\right) \to Y\!\left(\eta\!\left(W\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_W\!\left(\eta\right)\right)\right)\!\left([U,Y]_{VF}\right) \to [U,Y]_{VF}\!\left(\iota_W\!\left(\eta\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
[U,Y]_{VF}\!\left(\iota_W\!\left(\eta\right)\right) \to [U,Y]_{VF}\!\left(\eta\!\left(W\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_{[U,W]_{VF}}\!\left(\eta\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_{[U,W]_{VF}}\!\left(\eta\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_{[U,W]_{VF}}\!\left(\eta\right)\right) \to Y\!\left(\eta\!\left([U,W]_{VF}\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_W\!\left(L_U\!\left(\eta\right)\right)\right) \to Y\!\left(\left(L_U\!\left(\eta\right)\right)\!\left(W\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(L_U\!\left(\eta\right)\right)\!\left(W\right) \to U\!\left(\eta\!\left(W\right)\right) - \eta\!\left([U,W]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(\left(U\!\left(Y\!\left(\eta\!\left(W\right)\right)\right) - [U,Y]_{VF}\!\left(\eta\!\left(W\right)\right)\right) - Y\!\left(\eta\!\left([U,W]_{VF}\right)\right) - Y\!\left(U\!\left(\eta\!\left(W\right)\right) - \eta\!\left

In [8]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_U(d(ι_W(η)))(Y)
 -> (U(d(ι_W(η))(Y)) + (-d(ι_W(η))([U,Y]_VF)))

[2] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_W(η))(Y)
 -> Y(ι_W(η))

[3] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    Y(ι_W(η))
 -> Y(η(W))

[4] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_W(η))([U,Y]_VF)
 -> [U,Y]_VF(ι_W(η))

[5] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    [U,Y]_VF(ι_W(η))
 -> [U,Y]_VF(η(W))

[6] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_[U,W]_VF(η))(Y)
 -> Y(ι_[U,W]_VF(η))

[7] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    Y(ι_[U,W]_VF(η))
 -> Y(η([U,W]_VF))

[8] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_W(L_U(η))

### 3.T. Tilde analogue

$$
\tilde{\mathcal{L}}_\alpha\,\tilde{d}\,\tilde{\iota}_\gamma V
   - \tilde{d}\,\tilde{\iota}_{[\alpha,\gamma]_K} V
   - \tilde{d}\,\tilde{\iota}_\gamma\,\tilde{\mathcal{L}}_\alpha V \;=\; 0.
$$

We pick V as a **2-VF**; $\tilde{\iota}_\gamma V$ is 1-VF; $\tilde{d}$
turns it into 2-VF, and $\tilde{\mathcal{L}}_\alpha$ preserves
2-VF. Evaluating against $(\eta, \xi)$ drops all three terms to a
function. Thanks to the Phase 14.H polarity-flip extension the
engine recognises the SN-Jacobi obstruction wrapped under
$\tilde{d}$ in the form `MultiEval(V, d(⟨...⟩), γ)` and zeros it.


In [9]:
prob, Vmv, alpha, gamma = build_tilde(("α", "γ"), V_degree=2)
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))
xi_t = Symbol("ξ");  prob.registry.declare(xi_t, Graded(degree=1))
dt = TildeExteriorDerivative(prob.pi)

lhs = Sum.make(
    Act(TildeLieDerivative(alpha, prob.pi),
        Act(dt, Act(TildeInteriorProduct(gamma), Vmv))),
    Neg(Act(dt,
        Act(TildeInteriorProduct(prob.bracket(alpha, gamma)), Vmv))),
    Neg(Act(dt,
        Act(TildeInteriorProduct(gamma),
            Act(TildeLieDerivative(alpha, prob.pi), Vmv)))),
)
chain = prove_tilde(
    "(TIL-2) L̃_α d̃ ι̃_γ V − d̃ ι̃_[α,γ]_K V − d̃ ι̃_γ L̃_α V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t, xi_t))
display_chain(chain)


(TIL-2) L̃_α d̃ ι̃_γ V − d̃ ι̃_[α,γ]_K V − d̃ ι̃_γ L̃_α V = 0  [Poisson] -> closed in 199 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\left[\alpha,\, \gamma\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\left[\alpha,\, \gamma\right]_{[\cdot,\cdot]_K[\pi]}}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{\mathcal{L}}_{\alpha}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\alpha}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\alpha\right)\right)\!\left(\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\left[\alpha,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \left[\alpha,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{d}} intrinsic (Koszul) [\ensuremath{\pi}]: (\ensuremath{\tilde{d}}V)(\ensuremath{\eta}\_0, \ensuremath{\dots}) = \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(\ensuremath{\dots}) + \ensuremath{\Sigma} \ensuremath{\pm}V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\xi\right) \to V\!\left(\gamma,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\gamma}\!\left(V\right)\right)\!\left(\eta\right) \to V\!\left(\gamma,\, \eta\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}

In [10]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_α(d̃_π(ι̃_γ(V))) + (-d̃_π(ι̃_[·,·]_K[π](α, γ)(V))) + (-d̃_π(ι̃_γ(L̃_α(V)))))(η, ξ)
 -> (L̃_α(d̃_π(ι̃_γ(V)))(η, ξ) + (-d̃_π(ι̃_[·,·]_K[π](α, γ)(V))(η, ξ)) + (-d̃_π(ι̃_γ(L̃_α(V)))(η, ξ)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_α(d̃_π(ι̃_γ(V)))(η, ξ)
 -> (π♯(α)(d̃_π(ι̃_γ(V))(η, ξ)) + (-d̃_π(ι̃_γ(V))([·,·]_K[π](α, η), ξ)) + (-d̃_π(ι̃_γ(V))(η, [·,·]_K[π](α, ξ))))

[3] d̃ intrinsic (Koszul) [π]: (d̃V)(η_0, …) = Σ ±π^♯(η_i)·V(…) + Σ ±V([η_i,η_j]_K, …) [axiom]
    d̃_π(ι̃_γ(V))(η, ξ)
 -> (π♯(η)(ι̃_γ(V)(ξ)) + (-π♯(ξ)(ι̃_γ(V)(η))) + (-ι̃_γ(V)([·,·]_K[π](η, ξ))))

[4] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_γ(V)(ξ)
 -> V(γ, ξ)

[5] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_γ(V)(η)
 -> V(γ, η)

[6] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](η, ξ)
 -> (L_π♯(η)(ξ) + (-L_π♯(ξ)(η)) + (-d(⟨π♯(η), ξ⟩)))

[7] ι̃_ω intri

## 4. Identity 3: $\mathcal{L}_W\,d\,\iota_V \omega = d\,\iota_W\,d\,\iota_V \omega$

$$
\mathcal{L}_W\, d\,\iota_V \omega - d\,\iota_W\, d\,\iota_V \omega \;=\; 0.
$$

Derivation: Cartan magic gives $\mathcal{L}_W = d\,\iota_W + \iota_W\, d$;
setting $A := d\,\iota_V \omega$, we get
$\mathcal{L}_W A = d\,\iota_W A + \iota_W\,d\,A
   = d\,\iota_W\,d\,\iota_V \omega + \iota_W\,d^2\,\iota_V \omega
   = d\,\iota_W\,d\,\iota_V \omega + 0$.
So the identity really rests on $\iota_W\,d^2 = 0$.


In [11]:
lhs = Sum.make(
    multi_eval(Act(lie_derivative(W), Act(default_d, Act(interior(V_vf), omega))), Y),
    Neg(multi_eval(Act(default_d, Act(interior(W),
                Act(default_d, Act(interior(V_vf), omega)))), Y)),
)
chain = prove_std(
    "(STD-3) L_W d ι_V ω − d ι_W d ι_V ω = 0  on ω(Y)",
    lhs, Integer(0))
display_chain(chain)


(STD-3) L_W d ι_V ω − d ι_W d ι_V ω = 0  on ω(Y) -> closed in 12 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(L_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right)\!\left(Y\right) \to W\!\left(\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(Y\right)\right) - \left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left([W,Y]_{VF}\right) \quad \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, \ensuremath{\dots}) = X(\ensuremath{\omega}(Y\_1, \ensuremath{\dots})) \ensuremath{-} \ensuremath{\Sigma} \ensuremath{\omega}(\ensuremath{\dots}, [X, Y\_i]\_VF, \ensuremath{\dots})]\,(axiom)} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_V\!\left(\omega\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_V\!\left(\omega\right)\right) \to Y\!\left(\omega\!\left(V\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left([W,Y]_{VF}\right) \to [W,Y]_{VF}\!\left(\iota_V\!\left(\omega\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
[W,Y]_{VF}\!\left(\iota_V\!\left(\omega\right)\right) \to [W,Y]_{VF}\!\left(\omega\!\left(V\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right)\right)\!\left(Y\right) \to Y\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
Y\!\left(\iota_W\!\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\right) \to Y\!\left(\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(W\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(d\!\left(\iota_V\!\left(\omega\right)\right)\right)\!\left(W\right) \to W\!\left(\iota_V\!\left(\omega\right)\right) \quad \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, \ensuremath{\dots}, X\_p) = \ensuremath{\Sigma} \ensuremath{\pm}X\_i(\ensuremath{\omega}(\ensuremath{\dots},hat\_i,\ensuremath{\dots})) + \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\omega}([X\_i,X\_j]\_VF, \ensuremath{\dots})]\,(axiom)} \\
W\!\left(\iota_V\!\left(\omega\right)\right) \to W\!\left(\omega\!\left(V\right)\right) \quad \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) \ensuremath{\to} Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)} \\
\left(\left(W\!\left(Y\!\left(\omega\!\left(V\right)\right)\right) - [W,Y]_{VF}\!\left(\omega\!\left(V\right)\right)\right) - Y\!\left(W\!\left(\omega\!\left(V\right)\right)\right)\right) - 0 \to W\!\left(Y\!\left(\omega\!\left(V\right)\right)\right) - Y\!\left(W\!\left(\omega\!\left(V\right)\right)\right) - [W,Y]_{VF}\!\left(\omega\!\left(V\right)\right) \quad \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
W\!\left(Y\!\left(\omega\!\left(V\right)\right)\right) - Y\!\left(W\!\left(\omega\!\left(V\right)\rig

In [12]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] L_X intrinsic: (L_X ω)(Y_1, …) = X(ω(Y_1, …)) − Σ ω(…, [X, Y_i]_VF, …) [axiom]
    L_W(d(ι_V(ω)))(Y)
 -> (W(d(ι_V(ω))(Y)) + (-d(ι_V(ω))([W,Y]_VF)))

[2] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_V(ω))(Y)
 -> Y(ι_V(ω))

[3] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    Y(ι_V(ω))
 -> Y(ω(V))

[4] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_V(ω))([W,Y]_VF)
 -> [W,Y]_VF(ι_V(ω))

[5] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    [W,Y]_VF(ι_V(ω))
 -> [W,Y]_VF(ω(V))

[6] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_W(d(ι_V(ω))))(Y)
 -> Y(ι_W(d(ι_V(ω))))

[7] bare ι_X(ω) inside Act(D, _) → Act(D, MultiEval(ω, X)) [axiom]
    Y(ι_W(d(ι_V(ω))))
 -> Y(d(ι_V(ω))(W))

[8] d intrinsic (Koszul): (dω)(X_0, …, X_p) = Σ ±X_i(ω(…,hat_i,…)) + Σ ±ω([X_i,X_j]_VF, …) [axiom]
    d(ι_V(ω))

### 4.T. Tilde analogue

$$
\tilde{\mathcal{L}}_\gamma\,\tilde{d}\,\tilde{\iota}_\beta V
   - \tilde{d}\,\tilde{\iota}_\gamma\,\tilde{d}\,\tilde{\iota}_\beta V \;=\; 0.
$$

Same setup (V as 2-VF, $(\eta, \xi)$ evaluation): we need
$\tilde{d}^2 = 0$ (Aux-5, Poisson), tilde Cartan magic (14.F), and
the 14.G+H pipeline. The residue drops to the form $V(d(⟨...⟩),
\beta)$ under the $\tilde{d}$-wrapper, and the Phase 14.H
polarity-flip recognizer closes it.


In [13]:
prob, Vmv, beta, gamma = build_tilde(("β", "γ"), V_degree=2)
eta_t = Symbol("η"); prob.registry.declare(eta_t, Graded(degree=1))
xi_t = Symbol("ξ");  prob.registry.declare(xi_t, Graded(degree=1))
dt = TildeExteriorDerivative(prob.pi)

lhs = Sum.make(
    Act(TildeLieDerivative(gamma, prob.pi),
        Act(dt, Act(TildeInteriorProduct(beta), Vmv))),
    Neg(Act(dt, Act(TildeInteriorProduct(gamma),
            Act(dt, Act(TildeInteriorProduct(beta), Vmv))))),
)
chain = prove_tilde(
    "(TIL-3) L̃_γ d̃ ι̃_β V − d̃ ι̃_γ d̃ ι̃_β V = 0  [Poisson]",
    prob, lhs, Integer(0), etas=(eta_t, xi_t))
display_chain(chain)


(TIL-3) L̃_γ d̃ ι̃_β V − d̃ ι̃_γ d̃ ι̃_β V = 0  [Poisson] -> closed in 190 steps


{\allowdisplaybreaks\scriptsize
\begin{gather*}
\left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right) - \tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \quad \text{[MultiEval head linearity]\,(axiom)} \\
\left(\tilde{\mathcal{L}}_{\gamma}\!\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\gamma\right)\right)\!\left(\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right)\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\left[\gamma,\, \eta\right]_{[\cdot,\cdot]_K[\pi]},\, \xi\right) - \left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \left[\gamma,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{L}}\_\ensuremath{\omega} intrinsic [\ensuremath{\pi}]: (\ensuremath{\tilde{L}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = \ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\omega})\ensuremath{\cdot}V(\ensuremath{\eta}\_1, \ensuremath{\dots}) \ensuremath{-} \ensuremath{\Sigma} V(\ensuremath{\eta}\_1, \ensuremath{\dots}, [\ensuremath{\omega}, \ensuremath{\eta}\_i]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{d}\!\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\right)\!\left(\eta,\, \xi\right) \to \left(\pi\sharp\!\left(\eta\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\xi\right)\right) - \left(\pi\sharp\!\left(\xi\right)\right)\!\left(\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right)\right) - \left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]}\right) \quad \text{[\ensuremath{\tilde{d}} intrinsic (Koszul) [\ensuremath{\pi}]: (\ensuremath{\tilde{d}}V)(\ensuremath{\eta}\_0, \ensuremath{\dots}) = \ensuremath{\Sigma} \ensuremath{\pm}\ensuremath{\pi}^\ensuremath{\sharp}(\ensuremath{\eta}\_i)\ensuremath{\cdot}V(\ensuremath{\dots}) + \ensuremath{\Sigma} \ensuremath{\pm}V([\ensuremath{\eta}\_i,\ensuremath{\eta}\_j]\_K, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\xi\right) \to V\!\left(\beta,\, \xi\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right)\!\left(\eta\right) \to V\!\left(\beta,\, \eta\right) \quad \text{[\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} intrinsic: (\ensuremath{\tilde{\ensuremath{\iota}}}\_\ensuremath{\omega} V)(\ensuremath{\eta}\_1, \ensuremath{\dots}) = V(\ensuremath{\omega}, \ensuremath{\eta}\_1, \ensuremath{\dots})]\,(axiom)} \\
\left[\eta,\, \xi\right]_{[\cdot,\cdot]_K[\pi]} \to L_\pi\sharp(\eta)\!\left(\xi\right) - L_\pi\sharp(\xi)\!\left(\eta\right) - d\!\left(\langle \pi\sharp\!\left(\eta\right),\, \xi \rangle\right) \quad \text{[[\ensuremath{\alpha}, \ensuremath{\beta}]\_K = L\_\ensuremath{\rho}\ensuremath{\alpha}(\ensuremath{\beta}) \ensuremath{-} L\_\ensuremath{\rho}\ensuremath{\beta}(\ensuremath{\alpha}) \ensuremath{-} d\ensuremath{\langle}\ensuremath{\rho}\ensuremath{\alpha}, \ensuremath{\beta}\ensuremath{\rangle} [[\ensuremath{\cdot},\ensuremath{\cdot}]\_K[\ensuremath{\pi}]]]\,(axiom)} \\
\left(\tilde{\iota}_{\beta}\!\left(V\right)\right

In [14]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] MultiEval head linearity [axiom]
    (L̃_γ(d̃_π(ι̃_β(V))) + (-d̃_π(ι̃_γ(d̃_π(ι̃_β(V))))))(η, ξ)
 -> (L̃_γ(d̃_π(ι̃_β(V)))(η, ξ) + (-d̃_π(ι̃_γ(d̃_π(ι̃_β(V))))(η, ξ)))

[2] L̃_ω intrinsic [π]: (L̃_ω V)(η_1, …) = π^♯(ω)·V(η_1, …) − Σ V(η_1, …, [ω, η_i]_K, …) [axiom]
    L̃_γ(d̃_π(ι̃_β(V)))(η, ξ)
 -> (π♯(γ)(d̃_π(ι̃_β(V))(η, ξ)) + (-d̃_π(ι̃_β(V))([·,·]_K[π](γ, η), ξ)) + (-d̃_π(ι̃_β(V))(η, [·,·]_K[π](γ, ξ))))

[3] d̃ intrinsic (Koszul) [π]: (d̃V)(η_0, …) = Σ ±π^♯(η_i)·V(…) + Σ ±V([η_i,η_j]_K, …) [axiom]
    d̃_π(ι̃_β(V))(η, ξ)
 -> (π♯(η)(ι̃_β(V)(ξ)) + (-π♯(ξ)(ι̃_β(V)(η))) + (-ι̃_β(V)([·,·]_K[π](η, ξ))))

[4] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_β(V)(ξ)
 -> V(β, ξ)

[5] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_β(V)(η)
 -> V(β, η)

[6] [α, β]_K = L_ρα(β) − L_ρβ(α) − d⟨ρα, β⟩ [[·,·]_K[π]] [axiom]
    [·,·]_K[π](η, ξ)
 -> (L_π♯(η)(ξ) + (-L_π♯(ξ)(η)) + (-d(⟨π♯(η), ξ⟩)))

[7] ι̃_ω intrinsic: (ι̃_ω V)(η_1, …) = V(ω, η_1, …) [axiom]
    ι̃_β(V)((L_π

## Conclusion

The standard and tilde versions of the three derived identities
close syntactically via a single `prove_intrinsic_equivalence` /
`prob.prove_tilde_cartan` call. On the tilde side the closure of the
second and third identities required the polarity-flip extension
introduced in **Phase 14.H**: `TildeSnJacobiResidueDefinition` now
recognises both the canonical residue and its overall-sign-flipped
version.

| # | Standard | Tilde | $V$ | Eval | Poisson |
|---|---|---|---|---|---|
| 1 | $[\mathcal{L},\mathcal{L}]\mu = \mathcal{L}_{[\cdot]}\mu$, $\mu(Y)$ | $[\tilde{\mathcal{L}},\tilde{\mathcal{L}}] V = \tilde{\mathcal{L}}_{[\cdot]_K} V$ | 1-VF | $(\eta)$ | tilde |
| 2 | $\mathcal{L}\,d\iota - d\iota_{[\cdot]} - d\iota\,\mathcal{L}$ on $\eta(Y)$ | same, tilde | 2-VF | $(\eta, \xi)$ | tilde |
| 3 | $\mathcal{L}\,d\iota - d\iota\,d\iota$ on $\omega(Y)$ | same, tilde | 2-VF | $(\eta, \xi)$ | tilde |

All proofs are readable as LaTeX via `display_chain`; the
`Definition` from which each step originated is recorded in
`ProofStep.rule`.